# AQUAVIEW Python SDK — live demo

`pip install aquaview` — reach AQUAVIEW's ocean data from your own Python code
instead of hand-building API requests. Everything below runs against the **live**
API.

*Kernel: select the project `.venv`. For the keyed cells, set `AQUAVIEW_API_KEY`
first (in your terminal or a private cell).*

In [ ]:
import aquaview
print('SDK version:', aquaview.__version__)

## 1. Discover — no login needed

In [ ]:
client = aquaview.Client()

# What data sources can I query?
sources = client.get_sources()
print(len(sources), 'sources:', [s['source_id'] for s in sources])

# Search the catalog for a dataset
item = next(iter(client.search(limit=1, max_items=1).items()))
print('found dataset:', item.collection_id, '/', item.id)

## 2. Pull real data — one line

In [ ]:
# Slice the World Ocean Database to CSV — real temperatures at real coordinates.
print('pulling temperature from WOD ...\n')
csv_bytes = client.get_data(
    'wod', ['temperature', 'latitude', 'longitude'], limit=5, format='csv'
)
print(csv_bytes.decode())

## 3. With an API key — your account + the AI agent

`aquaview.Client()` picks up `AQUAVIEW_API_KEY` from the environment.

In [ ]:
import os
assert os.environ.get('AQUAVIEW_API_KEY'), 'Set AQUAVIEW_API_KEY first'
kc = aquaview.Client()

u = kc.get_usage()
print('plan    :', u['plan'])
print('storage :', u['usage']['storage_bytes'], '/', u['limits']['storage_bytes'], 'bytes')
print('sources :', [s['source_id'] for s in u['per_source']])

In [ ]:
# Ask the AQUAVIEW agent a question — returns the answer text.
print(kc.chat('what variables does the World Ocean Database have?'))

## 4. Large pulls — async export *(describe, don't run)*

For pulls too big to stream inline, the SDK runs them as a **background job**:

```python
job = kc.submit_export('gadr', ['temperature'])   # returns immediately
job.wait()                                         # poll until done
paths = job.download('out/')                       # one file per time shard
```

Finished and verified in tests — it goes live once the paired API change deploys.